# **DeepLIFT дальше: RevealCancel и DeepSHAP**

Практика к модулю [«Атрибуция от аксиом»](https://ai-interpretability.school).

Урок разобрал два продолжения DeepLIFT и оба оставил на словах. Здесь мы их посчитаем.

**RevealCancel.** Урок приводит для него точный пример — нейрон $h=\mathrm{ReLU}(i-j)$ при
эталоне $i^0=j^0=0$ и входе $i=j=5$ — и объясняет словами, почему Rescale там слеп. Пример
считается в пять строк, и счет убеждает сильнее объяснения.

**DeepSHAP.** Урок говорит: усредните DeepLIFT по эталонам из данных — получите приближение
значений Шепли. Проверим, что при этом происходит с суммой атрибуций.

Считает мгновенно.

In [ ]:
import torch
import torch.nn as nn

def g(v):
    """ReLU для одного числа — она понадобится обоим правилам."""
    return max(0.0, float(v))


def rescale(z0, dz_pos, dz_neg):
    """Rescale: одна секущая на суммарное отклонение входа."""
    dz = dz_pos + dz_neg
    if abs(dz) < 1e-9:
        return 0.0, 0.0                 # вырожденный случай: отклонение входа нулевое
    m = (g(z0 + dz) - g(z0)) / dz
    return m * dz_pos, m * dz_neg


def reveal_cancel(z0, dz_pos, dz_neg):
    """RevealCancel: положительная и отрицательная части идут через нелинейность порознь."""
    dg_pos = 0.5 * ((g(z0 + dz_pos) - g(z0))
                    + (g(z0 + dz_neg + dz_pos) - g(z0 + dz_neg)))   # положительная часть: сначала одна, потом после отрицательной
    dg_neg = 0.5 * ((g(z0 + dz_neg) - g(z0))
                    + (g(z0 + dz_pos + dz_neg) - g(z0 + dz_pos)))   # отрицательная часть: зеркально
    return dg_pos, dg_neg


def show(title, z0, dz_pos, dz_neg):
    dh = g(z0 + dz_pos + dz_neg) - g(z0)
    print(f'{title}   отклонение выхода {dh:+.2f}')
    for name, (a, b) in (('Rescale', rescale(z0, dz_pos, dz_neg)),
                         ('RevealCancel', reveal_cancel(z0, dz_pos, dz_neg))):
        print(f'   {name:14} i {float(a):+.2f}   j {float(b):+.2f}   сумма {float(a)+float(b):+.2f}')

## 1. Полное гашение: то самое место, где Rescale слеп

Вход $i$ толкает нейрон вверх на 5, вход $j$ — вниз на 5. Выход не сдвинулся ни на сколько.

In [ ]:
show(f'Полное гашение: i = 5, j = 5', z0=0.0, dz_pos=5.0, dz_neg=-5.0)

**Rescale выдал ноль обоим.** И это не ошибка: он честно докладывает, что выход
не изменился. Но всю механику происходящего он при этом скрыл — по его карте нельзя отличить
«оба входа не важны» от «оба важны и погасили друг друга».

**RevealCancel выдал $+2{,}50$ и $-2{,}50$.** Он провел положительную и отрицательную части
через нелинейность порознь и увидел, что там происходило.

Обратите внимание на суммы: **у обоих ноль**. Свойство summation-to-delta не нарушено ни тем,
ни другим — просто одна и та же сумма разложена по-разному. Это важно: RevealCancel не чинит
сломанную аксиому, он вскрывает то, что аксиома допускает.

**Задание 1.** Проверьте, что произойдет, если поменять местами роли: $i=-5$, $j=-5$, то есть
$\Delta z^{+}=5$ станет $\Delta z^{-}$ и наоборот. Симметричен ли RevealCancel?

In [ ]:
# Ваш код здесь

## 2. А если гашение неполное

Урок говорит, что при $j=4$ «нейрон мгновенно оживет». Посмотрим, что скажут оба правила.

In [ ]:
show(f'Неполное гашение: i = 5, j = 4', z0=0.0, dz_pos=5.0, dz_neg=-4.0)

Вот это стоит рассмотреть внимательно — **урок про такое не говорит вовсе.**

Здесь Rescale уже не слеп: выход сдвинулся на 1, и правило разложило этот сдвиг как
$+5$ и $-4$ — то есть приписало входам их полные отклонения. RevealCancel дал $+3$ и $-2$.

**Обе раскладки суммируются в одно и то же $+1$**, и обе законны. Но говорят они разное.
Rescale: «$i$ толкнул на 5, $j$ вернул на 4». RevealCancel усредняет по двум порядкам
добавления и потому мягче: часть эффекта каждого входа он относит на их совместное действие,
а не на них поодиночке.

Вывод, который стоит унести: **правила расходятся не только в вырожденном случае**. Выбор
между ними — содержательный, а не аварийный, и объявлять его надо всегда, а не только когда
что-то сломалось.

**Задание 2.** Постройте обе раскладки для $j$ от 0 до 5 с шагом 1 и нарисуйте две кривые
вклада $i$. В какой точке они расходятся сильнее всего и почему именно там?

In [ ]:
# Ваш код здесь

## 3. DeepSHAP: тот же прием, что у Expected Gradients

Урок обращает внимание: Expected Gradients относятся к Integrated Gradients ровно так же, как
DeepSHAP к DeepLIFT. Один фиксированный эталон заменяется распределением эталонов.

Сначала посчитаем DeepLIFT с одним нулевым эталоном — как в прошлых уроках.

In [ ]:
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(4, 6, bias=False), nn.ReLU(),
                    nn.Linear(6, 3, bias=False)).eval()
TARGET = 1


def trace(x):
    acts = [x]
    for layer in net:
        x = layer(x)
        acts.append(x)
    return acts


def deeplift(x, baseline, c=TARGET):
    """DeepLIFT (Rescale) на сети: множители, цепное правило, отклонение входа."""
    a_x, a_0 = trace(x), trace(baseline)
    m = torch.zeros_like(a_x[-1])
    m[0, c] = 1.0
    for i in range(len(net) - 1, -1, -1):
        layer = net[i]
        if isinstance(layer, nn.Linear):
            m = m @ layer.weight
        else:
            dz, dx = a_x[i + 1] - a_0[i + 1], a_x[i] - a_0[i]
            safe = torch.where(dx.abs() > 1e-7, dx, torch.ones_like(dx))
            m = m * torch.where(dx.abs() > 1e-7, dz / safe, torch.zeros_like(dx))
    return (m * (x - baseline)).detach()


torch.manual_seed(1)
data = torch.rand(200, 4)          # обучающая выборка — она же распределение эталонов
x = torch.rand(1, 4) + 0.3

single = deeplift(x, torch.zeros(1, 4))
print(f'один нулевой эталон: {single.numpy().round(4)}   сумма {single.sum():+.4f}')

Теперь заменим один эталон распределением: возьмем эталоны из данных и усредним
атрибуции по ним.

In [ ]:
for n in (1, 5, 25, 200):
    attributions = torch.stack([deeplift(x, data[i:i + 1]) for i in range(n)]).mean(0)
    mean_out = torch.stack([net(data[i:i + 1])[0, TARGET] for i in range(n)]).mean()
    delta = (net(x)[0, TARGET] - mean_out).item()
    print(f'усреднение по эталонам, штук {n:3}: {attributions.numpy().round(4)}   сумма {attributions.sum():+.4f}   '
          f'f(x) - среднее f(эталон): {delta:+.4f}')

Смотрите на две последние колонки каждой строки: **сумма атрибуций совпадает
с разностью между выходом на объекте и средним выходом на эталонах — на каждом числе эталонов,
до четвертого знака.**

Это и есть summation-to-delta, только $\Delta t$ теперь считается не от одной точки, а от
среднего по распределению. Свойство пережило замену эталона на распределение, и в этом весь
смысл приема: мы сняли произвол выбора точки, ничего не потеряв из гарантий.

Заодно видно, как атрибуции успокаиваются с ростом числа эталонов: от одного к пяти они
меняются заметно, от двадцати пяти к двумстам — уже почти нет.

**Задание 3.** Сравните атрибуции DeepSHAP с атрибуциями Expected Gradients на том же объекте
и том же наборе эталонов (код Expected Gradients — в тетради «Expected Gradients» этого же
модуля). Насколько они близки? Урок утверждает, что оба приближают одно и то же — проверьте.

In [ ]:
# Ваш код здесь

## Что унести из тетради

- **Rescale слеп к взаимоуничтожению, и это видно числом.** При полном гашении он выдает ноль
  обоим входам, RevealCancel — $+2{,}5$ и $-2{,}5$. Обе раскладки суммируются в ноль: аксиома
  не нарушена ни одним из правил, разница в том, что одно правило показывает механику, а
  другое нет.
- **Правила расходятся и при неполном гашении.** Выбор между Rescale и RevealCancel —
  содержательное решение, которое надо объявлять рядом с картой, а не молчаливая деталь
  реализации.
- **DeepSHAP сохраняет summation-to-delta**, только отклонение теперь считается от среднего
  по распределению эталонов. Замена точки на распределение убирает произвол, не теряя гарантии.
- **Прием один и тот же в двух семействах:** Expected Gradients относятся к Integrated
  Gradients так же, как DeepSHAP к DeepLIFT.